# Lesson 5.3: Confidence Intervals for a Proportion

Build 95% confidence intervals for a proportion with the Wald and Wilson methods, and see why Wilson is safer for small samples.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

## The question

A polling firm asked 850 voters and 476 support Candidate A. A clinic also tried a new procedure on 20 patients and 3 had a complication. We want an honest 95% interval for each proportion, and need to see which method (the simple Wald or the Wilson interval) can be trusted when the sample is small.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## Step 1: A healthy poll: 476 of 850

The lesson's poll. statsmodels' proportion_confint takes the count of successes, the sample size, and a method: 'normal' is the classic Wald interval, 'wilson' is the Wilson score interval.

In [1]:
from statsmodels.stats.proportion import proportion_confint  # Load the proportion interval function from statsmodels.

x, n = 476, 850  # Store the poll: 476 supporters out of 850 voters.
print(f'p-hat = {x / n:.3f}')  # Print the sample proportion p-hat = x / n.
for method in ('normal', 'wilson'):  # Loop over two methods: 'normal' is the Wald interval and 'wilson' is the Wilson interval.
    low, high = proportion_confint(x, n, alpha=0.05, method=method)  # Build the 95% interval (alpha = 0.05 means 5% in the tails) with this method.
    print(f'{method:7s}: ({low:.4f}, {high:.4f})')  # Print the method name and its two limits.

p-hat = 0.560
normal : (0.5266, 0.5934)
wilson : (0.5264, 0.5930)


**What this means.** The poll has p-hat = 0.560. The Wald interval is (0.5266, 0.5934) and the Wilson interval is (0.5264, 0.5930), practically identical. With a big sample and a mid-range proportion both methods work, and the whole interval is above 0.5.

## Step 2: A small sample: 3 of 20

Now only 3 of 20 patients had a complication. n times p-hat is just 3, below the rule of 10, so the Wald interval is not trustworthy here. The raw Wald formula gives a lower limit below 0 (about -0.006), which is impossible, so statsmodels cuts it off at 0.

In [2]:
x, n = 3, 20  # Switch to the clinic data: 3 complications out of 20 patients.
print(f'p-hat = {x / n:.2f}')  # Print p-hat.
for method in ('normal', 'wilson'):  # Loop over the same two methods.
    low, high = proportion_confint(x, n, alpha=0.05, method=method)  # Build the 95% interval with each.
    print(f'{method:7s}: ({low:.3f}, {high:.3f})')  # Print the limits.

p-hat = 0.15
normal : (0.000, 0.306)
wilson : (0.052, 0.360)


**What this means.** p-hat = 0.15. Wald gives (0.000, 0.306): its raw lower limit is about -0.006, impossible for a proportion, so statsmodels cuts it at 0. Wilson gives (0.052, 0.360), staying inside 0 to 1. Here n*p-hat is only 3, so Wald is unreliable and Wilson is the safer choice.

## Step 3: No events at all: 0 of 20

With zero events, Wald collapses to (0, 0), which claims we are certain the rate is zero after only 20 patients. Wilson gives a sensible upper limit.

In [3]:
for method in ('normal', 'wilson'):  # Try the extreme case: loop over both methods.
    low, high = proportion_confint(0, 20, alpha=0.05, method=method)  # Build the interval for zero complications in 20 patients.
    print(f'{method:7s}: ({low:.3f}, {high:.3f})')  # Print the limits.

normal : (0.000, 0.000)
wilson : (0.000, 0.161)


**What this means.** Wald gives (0.000, 0.000), claiming we are certain the complication rate is zero after only 20 patients, which is absurd. Wilson gives (0.000, 0.161), saying the true rate could plausibly be as high as 16%.

## Step 4: Coverage test

Suppose the true rate is exactly 10% and n = 20. We simulate 20,000 samples, build both intervals each time and count how often they contain 10%. A good 95% interval should be close to 95%.

In [4]:
import numpy as np  # Load NumPy for random numbers.

rng = np.random.default_rng(3)  # Create a seeded random generator.
counts = rng.binomial(20, 0.10, size=20_000)  # Simulate 20,000 clinics of 20 patients each, with a true complication rate of exactly 10%.
for method in ('normal', 'wilson'):  # Loop over the two methods.
    low, high = proportion_confint(counts, 20, alpha=0.05, method=method)  # Build an interval for each of the 20,000 samples at once.
    covered = (low <= 0.10) & (0.10 <= high)  # Check which intervals contain the true 10%.
    print(f'{method:7s}: covers the true 10% in {covered.mean():.1%} of samples')  # Print the share of samples whose interval covers the truth.

normal : covers the true 10% in 87.5% of samples
wilson : covers the true 10% in 95.7% of samples


**What this means.** A 95% interval should catch the truth in 95% of samples. Wald catches the true 10% in only 87.5%, a hidden 7.5-point shortfall, while Wilson catches it in 95.7%. This is why Wilson is the better default for small samples.

## So, what is the answer?

The poll shows 56.0% support with a 95% interval of about 52.7% to 59.3%, so a majority is clearly in favour. The clinic's 3 of 20 gives a Wilson interval of 5.2% to 36.0%; the Wald interval wrongly extends below 0 and covers the truth only 87.5% of the time. Caution: small samples give wide intervals, whatever method you choose.

## Now you try

An online shop tested a new checkout page. Out of 40 visitors, 6 completed a purchase. Separately, 0 of 25 visitors used a new gift-wrap option. A third sample had 180 of 400 clicking a banner. Compare Wald and Wilson for all three.

**Data:** 6 of 40, 0 of 25 and 180 of 400 (x successes, n visitors).

1. Compute p-hat for each sample.
2. Compute the 95% Wald and Wilson intervals for each.
3. Where do the two methods differ most, and why?

*Hint:* Use proportion_confint(x, n, alpha=0.05, method='normal') and method='wilson' from statsmodels.

In [ ]:
# Your turn: write your code here


### Solution (try first, then peek)

In [ ]:
from statsmodels.stats.proportion import proportion_confint
for x, n in ((6, 40), (0, 25), (180, 400)):
    print(f'{x} of {n}: p-hat = {x / n:.3f}')
    for method in ('normal', 'wilson'):
        low, high = proportion_confint(x, n, alpha=0.05, method=method)
        print(f'   {method:7s} ({low:.3f}, {high:.3f})')

6 of 40: p-hat = 0.150
   normal  (0.039, 0.261)
   wilson  (0.071, 0.291)
0 of 25: p-hat = 0.000
   normal  (0.000, 0.000)
   wilson  (0.000, 0.133)
180 of 400: p-hat = 0.450
   normal  (0.401, 0.499)
   wilson  (0.402, 0.499)


**Answer.** For 6 of 40 (p-hat 0.150), Wald is (0.039, 0.261) and Wilson (0.071, 0.291). For 0 of 25, Wald collapses to (0.000, 0.000) while Wilson gives (0.000, 0.133). For 180 of 400 (0.450) they agree: (0.401, 0.499) and (0.402, 0.499). They differ when the sample is small or the rate is extreme, so use Wilson there.

## The same question in R, SPSS and Stata

**R**

```r
x <- 3; n <- 20; phat <- x / n  # Clinic data: 3 of 20, and p-hat.
me <- 1.96 * sqrt(phat * (1 - phat) / n)  # Wald margin of error by hand: 1.96 times the SE.
c(phat - me, phat + me)  # Print the Wald limits.
prop.test(x, n, correct = FALSE)$conf.int  # Wilson interval: prop.test with correct = FALSE.
binom.test(x, n)$conf.int  # Exact interval from binom.test.
prop.test(476, 850, correct = FALSE)$conf.int  # Wilson interval for the poll (476 of 850).
```

How to read it: The Wald limits print about -0.006 and 0.306 (negative, impossible). The Wilson interval is 0.0524 to 0.3604, the exact interval 0.0321 to 0.3789, and the poll's Wilson interval 0.5264 to 0.5930.

**SPSS** (Transform > Compute Variable (SPSS menus offer exact and Jeffreys intervals under Analyze > Nonparametric Tests > One Sample, but not Wald or Wilson))

```
DATA LIST FREE / x n.
BEGIN DATA
3 20
END DATA.
COMPUTE phat = x / n.
COMPUTE z = IDF.NORMAL(0.975, 0, 1).
COMPUTE wlo = phat - z * SQRT(phat * (1 - phat) / n).
COMPUTE whi = phat + z * SQRT(phat * (1 - phat) / n).
COMPUTE ctr = (phat + z**2 / (2 * n)) / (1 + z**2 / n).
COMPUTE half = z * SQRT(phat * (1 - phat) / n + z**2 / (4 * n**2)) / (1 + z**2 / n).
COMPUTE slo = ctr - half.
COMPUTE shi = ctr + half.
LIST VARIABLES=wlo whi slo shi.
```

How to read it: The listing shows Wald lower -0.006 and upper 0.306 (negative, so impossible), and Wilson 0.052 and 0.360. For the poll change the data row to 476 850 and rerun.

**Stata**

```
cii proportions 20 3, wald
cii proportions 20 3, wilson
cii proportions 20 3, exact
cii proportions 850 476, wald
cii proportions 850 476, wilson
```

How to read it: Look at the Lb and Ub (lower and upper bound) columns. For 3 of 20 Wilson gives about 0.052 to 0.360 and Exact about 0.032 to 0.379. For the poll both Wald and Wilson are near 0.527 to 0.593.